# LatentMind V6 — Colab Notebook

**Training-first pipeline**: generates planner dataset → trains MLP head → loads agent → runs tests.

Requires: T4 or L4 GPU runtime.

In [ ]:
%%capture
!pip install -q \
    transformers==4.46.3 \
    sentence-transformers==3.3.1 \
    langgraph==0.2.53 \
    langchain-core==0.3.25 \
    bitsandbytes==0.44.1 \
    accelerate==1.2.1 \
    scipy matplotlib jinja2 pydantic pymysql
print("deps installed")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = "https://github.com/Hamza09Hamza/Latent-Djezzy.git"
REPO_DIR = "/content/Latent-Djezzy"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
    print(f"cloned → {REPO_DIR}")
else:
    # always pull to get the latest code
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    # clear cached v6 modules so reimports get fresh code
    for mod in list(sys.modules.keys()):
        if "v6" in mod:
            del sys.modules[mod]
    print(f"pulled latest code, cleared module cache")

os.chdir(REPO_DIR)
print("repo ready:", REPO_DIR)

In [ ]:
import shutil, os

# Adjust DRIVE_DB to where interndb.sqlite lives on YOUR Drive.
DRIVE_DB   = "/content/drive/MyDrive/interndb.sqlite"
LOCAL_DB   = "/content/interndb.sqlite"

if not os.path.isfile(LOCAL_DB):
    shutil.copy(DRIVE_DB, LOCAL_DB)
    print(f"copied {os.path.getsize(LOCAL_DB):,} bytes → {LOCAL_DB}")
else:
    print("SQLite already present:", LOCAL_DB)

In [ ]:
import os

os.environ["V6_USE_SQLITE"]    = "1"
os.environ["V6_SQLITE_PATH"]   = "/content/interndb.sqlite"
os.environ["V6_4BIT"]          = "1"          # 4-bit NF4 → fits 3B on T4
os.environ["V6_SLM_OVERRIDE"]  = "Qwen/Qwen2.5-Coder-3B-Instruct"
os.environ["V6_PLANNER"]       = "prototype"  # overridden to mlp after training

import torch
print("CUDA:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
# Generate synthetic labelled examples for the planner MLP head.
# Output: v6/data/planner_train.jsonl  (~1 273 examples)
!python3 -m v6.planner_data
!wc -l v6/data/planner_train.jsonl

In [ ]:
# Train 1024→256→(intent, caps) head — ~80 epochs, <5 min on T4.
# Output: models/planner_head.pt
!python3 -m v6.train_planner

import os
os.environ["V6_PLANNER"] = "mlp"
print("Planner mode set to MLP.")

In [ ]:
import sys, os
sys.path.insert(0, "/content/Latent-Djezzy")

from v6.graph import LatentMindV6

print("Loading LatentMind V6 agent (this downloads/loads the SLM + BGE-M3)…")
agent = LatentMindV6()
print("Agent ready.")

In [ ]:
import time

def ask(question: str, thread: str = "main") -> None:
    """Stream node-by-node progress, then print the final answer."""
    config = {"configurable": {"thread_id": thread}}
    state  = {"messages": [{"role": "user", "content": question}]}

    final_answer = ""
    t0 = time.time()

    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")

    for event in agent.graph.stream(state, config=config, stream_mode="updates"):
        for node_name, data in event.items():
            if node_name == "plan":
                ps    = data.get("plan_scores") or {}
                score = ps.get("score", 0.0)
                intent = data.get("intent", "?")
                caps   = data.get("capabilities", [])
                print(f"  [plan]   intent={intent}  conf={score:.2f}  caps={caps}")

            elif node_name == "sql_generate":
                sql = data.get("sql", "")
                if sql:
                    short = sql.replace("\n", " ")[:80]
                    print(f"  [sql]    {short}…" if len(sql) > 80 else f"  [sql]    {short}")

            elif node_name == "sql_execute":
                rows = data.get("rows") or []
                err  = data.get("sql_error", "")
                if err:
                    print(f"  [exec]   ERROR: {err[:60]}")
                else:
                    print(f"  [exec]   {len(rows)} row(s)")

            elif node_name == "answer":
                final_answer = data.get("answer", "")

            elif node_name == "direct_answer":
                final_answer = data.get("answer", "")

            elif node_name == "finalize":
                if not final_answer:
                    final_answer = data.get("answer", "")

            else:
                pass  # route/retrieve/orchestrate/etc. — silent

    elapsed = time.time() - t0
    print(f"\nAnswer ({elapsed:.1f}s):")
    print(final_answer or "(no answer captured)")
    print()

In [ ]:
ask("Hello, what can you do?")

In [ ]:
ask("What is ARPU?")

In [ ]:
ask("What is the total revenue for Oran in 2024?")

In [ ]:
ask("Compare churn rates between Algiers and Constantine last quarter")

In [ ]:
ask("Show me a bar chart of monthly revenue by wilaya for Q1 2024")

In [ ]:
ask("Generate an executive report for Q4 2024 performance")

In [ ]:
ask("Which wilaya had the highest churn rate?")  # follow-up — tests memory

In [ ]:
import torch
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated()  / 1e9
    reserv = torch.cuda.memory_reserved()   / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  allocated: {alloc:.2f} GB")
    print(f"GPU  reserved:  {reserv:.2f} GB")
    print(f"GPU  total:     {total:.2f} GB")

In [ ]:
# Direct streaming — bypasses the graph, useful for quick SLM checks.
from v6.slm import get_slm

slm = get_slm()
messages = [
    {"role": "system", "content": "You are a helpful telecom analyst."},
    {"role": "user",   "content": "List the top 3 KPIs for a telecom operator."},
]
print("Streaming response:")
for token in slm.stream_generate(messages, max_new_tokens=256):
    print(token, end="", flush=True)
print()